# TSMC SEC 6-K 2011–2026 元数据索引

冻结两次 SEC submissions GET，只核对年度与 accession 覆盖，不读市场结果或拟合。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_tsmc_sec_6k_guidance_source_probe_20261023';raw=root/'data/snapshots/stage1_tsmc_sec_6k_guidance_source_probe_20261023';plan=json.loads((card/'submissions_index_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(card/'result.json')==plan['parent_body_result_sha256'] and sha(card/'acceptance_clock_result.json')==plan['parent_acceptance_result_sha256'] and not (card/'submissions_index_state.json').exists()
state={'status':'started','GET_attempts':0,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'submissions_index_state.json').write_text(json.dumps(state,indent=2)+'\n');receipts=[];documents={};failure=None
for i,spec in enumerate(plan['exact_requests'],1):
 body=raw/f'submissions_{spec["name"]}.json';headers=raw/f'submissions_{spec["name"]}.headers';state['GET_attempts']+=1;(card/'submissions_index_state.json').write_text(json.dumps(state,indent=2)+'\n');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time','60','--max-filesize','8000000','--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['request']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);parts=run.stdout.rsplit(' ',1);status=int(parts[0]) if len(parts)==2 and parts[0].isdigit() else None
  receipt={'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':parts[-1] if parts else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'exception':str(exc)[:300]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None)
 receipts.append(receipt);(card/'submissions_capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists():failure=f'{spec["name"]}_transport';break
 try:
  d=json.loads(body.read_text());source=d['filings']['recent'] if spec['name']=='recent' else d
  if spec['name']=='recent':assert str(d['cik']) in ('0001046179','1046179') and any(x['name']=='CIK0001046179-submissions-001.json' for x in d['filings']['files'])
  keys=['accessionNumber','filingDate','form','primaryDocument','acceptanceDateTime'];n=len(source['accessionNumber']);assert n>100 and all(len(source[k])==n for k in keys)
  documents[spec['name']]=source
 except Exception as exc:failure=f'{spec["name"]}_schema_{type(exc).__name__}_{str(exc)[:120]}';break
 time.sleep(plan['request']['seconds_between_requests'])
files=[{'name':x.name,'sha256':sha(x),'bytes':x.stat().st_size} for x in sorted(raw.iterdir()) if x.name.startswith('submissions_')];manifest={'plan_sha256':sha(card/'submissions_index_plan.json'),'files':files,'receipts_sha256':sha(card/'submissions_capture_receipts.json')};(card/'submissions_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='tsmc-sec6k-index-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in files:t.add(raw/x['name'],arcname='raw/'+x['name'])
  for name in ('submissions_index_plan.json','submissions_capture_receipts.json','submissions_manifest.json'):t.add(card/name,arcname=name)
 digest=sha(archive);a=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert a['sha256']==digest and a['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='tsmc-sec6k-index-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in files:
    m=t.extractfile('raw/'+x['name']);assert m and hashlib.sha256(m.read()).hexdigest()==x['sha256']
counts={};total=0;forms=0;quarter_month=0;unique=set()
for name,src in documents.items():
 for accession,day,form in zip(src['accessionNumber'],src['filingDate'],src['form']):
  assert accession not in unique;unique.add(accession);total+=1
  if form.replace('FORM ','')=='6-K':
   forms+=1;year=day[:4];counts[year]=counts.get(year,0)+1
   if day[5:7] in ('01','04','07','10'):quarter_month+=1
result={'status':'official_SEC_TSMC_2011_2026_accession_index_complete' if not failure and len(documents)==2 and all(str(y) in counts for y in range(2011,2027)) and forms>=50 else 'bounded_SEC_TSMC_index_partial_or_failed','failure':failure,'plan_sha256':sha(card/'submissions_index_plan.json'),'GET_attempts':state['GET_attempts'],'total_accessions':total,'6K_count':forms,'6K_by_year':counts,'6K_in_calendar_quarter_reporting_months':quarter_month,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'all_quarter_guidance_documents_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'submissions_index_result.json').write_text(json.dumps(result,indent=2)+'\n');state.update(status='completed' if result['status'].startswith('official_') else 'partial_or_failed',NAS_archive_sha256=digest);(card/'submissions_index_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'6K':forms,'years':len(counts),'NAS':digest})


{'status': 'official_SEC_TSMC_2011_2026_accession_index_complete', 'failure': None, 'GETs': 2, '6K': 1334, 'years': 25, 'NAS': '12d25b1d616498b01d2deb77a0aaec0b24d16800baaf7bae458b9ea90461d023'}


## 中文结论

SEC 两份提交索引覆盖 2011–2026，每年有 6-K；原件已备份 NAS。 详见 `conclusion.md`。